# compare_runs

Paired comparison of any two scored runs on the same DATASET. Joins on uid, not position.

Threshold is computed from **this contrast's own sigma_d**, not from a constant. sigma_d is not a fixed property of the pipeline: an identical config re-run gives 0.966, and the effect of a prompt change depends on the contrast: the positive-prefix arms and n_empty sit at ~1.2-1.4 because the text re-rolls each row, while the appended-negative arms stay near the floor at 1.016-1.066. Quoting the identical-config floor as a detection threshold understates it by ~35%.

Reports the difference test and the pre-registered equivalence test (TOST, +/-0.15) side by side. Three outcomes: differ, equivalent, inconclusive. The bound was fixed before any arm was generated and is not to be adjusted here.

Reference constants, measured: judge sigma_item 0.362 (RMS over sd_reps, same image x5); identical-config sigma_d 0.966 (screen, 302 rows, cross-session); generation-only sigma 0.58.

## 1. Config

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")

DRIVE   = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET = "screen"          # screen | test
RUN_A   = "a1_complete"     # the condition under test
RUN_B   = "a5_filler"       # the comparator: baseline for a raw effect

EQUIV_BOUND = 0.15          # pre-registered before any arm was generated; do not tune
SIGMA_ITEM  = 0.362         # judge re-scoring SD on an identical image (sd_reps RMS)
SIGMA_D_REF = 0.966         # identical-config sigma_d (screen 302, cross-session)

for r in (RUN_A, RUN_B):
    p = f"{DRIVE}/scores/{DATASET}__{r}.jsonl"
    assert os.path.exists(p), f"missing: {p}"
print(f"{DATASET}: {RUN_A}  vs  {RUN_B}")

Mounted at /content/drive
screen: a1_complete  vs  a5_filler


## 2. Load, join on uid

In [ ]:
import json, math
from collections import defaultdict

def load(run):
    d, errs = {}, defaultdict(int)
    for l in open(f"{DRIVE}/scores/{DATASET}__{run}.jsonl", encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: errs[r["error"]] += 1; continue
        d[r["i"]] = r
    return d, errs

A, ea = load(RUN_A)
B, eb = load(RUN_B)
I = sorted(set(A) & set(B))

# a row index means nothing on its own; the uid is what identifies the data row
bad = [i for i in I if A[i]["uid"] != B[i]["uid"]]
assert not bad, f"uid mismatch on rows {bad[:5]} — runs are not aligned"

print(f"{RUN_A}: {len(A)} scored" + (f" | errors {dict(ea)}" if ea else ""))
print(f"{RUN_B}: {len(B)} scored" + (f" | errors {dict(eb)}" if eb else ""))
print(f"paired on {len(I)} rows, uid-verified")

for run, m in [(RUN_A, f"{DRIVE}/gen/{DATASET}/{RUN_A}/manifest.json"),
               (RUN_B, f"{DRIVE}/gen/{DATASET}/{RUN_B}/manifest.json")]:
    if os.path.exists(m):
        j = json.load(open(m))
        print(f"  {run:14s} {j.get('gpu')} | {str(j.get('instruction_row0'))[:70]}...")

a1_complete: 302 scored
a5_filler: 302 scored
paired on 302 rows, uid-verified
  a1_complete    NVIDIA A100-SXM4-80GB | Every object and person referenced below appears in the final image, c...
  a5_filler      NVIDIA A100-SXM4-80GB | The text that follows specifies an image to be generated using the ref...


## 3. Difference test, with the threshold this contrast actually achieves

In [ ]:
import statistics as st
from scipy.stats import ttest_rel, chi2, norm, f as fdist

ov = lambda d: math.sqrt(max(d["PF"], 0) * max(d["SC"], 0))
n  = len(I)
Z  = norm.ppf(0.975) + norm.ppf(0.80)      # 1.96 + 0.84: 5% false-positive, 80% power

print(f"{'metric':8s} {'mean':>8s} {'SD':>7s} {'SE':>7s} {'95% CI':>18s} {'t':>7s} {'p':>7s}")
for lab, fn in [("PF", lambda d: d["PF"]), ("SC", lambda d: d["SC"]), ("Overall", ov)]:
    d  = [fn(A[i]) - fn(B[i]) for i in I]
    se = st.stdev(d) / math.sqrt(n)
    t, p = ttest_rel([fn(A[i]) for i in I], [fn(B[i]) for i in I])
    lo, hi = st.mean(d) - 1.96*se, st.mean(d) + 1.96*se
    print(f"{lab:8s} {st.mean(d):>+8.3f} {st.stdev(d):>7.3f} {se:>7.3f} {'[%+.3f, %+.3f]'%(lo,hi):>18s} {t:>+7.2f} {p:>7.3f}")

d  = [ov(A[i]) - ov(B[i]) for i in I]
sd = st.stdev(d)
df = n - 1
sd_lo = sd*math.sqrt(df/chi2.ppf(0.975, df)); sd_hi = sd*math.sqrt(df/chi2.ppf(0.025, df))
thr = Z * sd / math.sqrt(n)

print(f"\nsigma_d (Overall)      = {sd:.3f}   95% CI [{sd_lo:.3f}, {sd_hi:.3f}]")
print(f"identical-config floor = {SIGMA_D_REF:.3f}   (re-running the SAME config)")
print(f"judge-only floor       = {math.sqrt(2)*SIGMA_ITEM:.3f}   (sqrt(2) x {SIGMA_ITEM})")
# Does this contrast scatter rows beyond what re-running the SAME config already does?
# Both delta series share a run, so the two sigma_d estimates are correlated -- an independent
# F-test does not apply. The deltas are also heavy-tailed (Shapiro-Wilk p<1e-15, excess kurtosis
# 8-19), which rules out Pitman-Morgan as well; both parametric tests overstate significance by
# 2.6-8.4 orders of magnitude here. Paired bootstrap on the variance ratio, resampling rows so the
# X/Y pairing survives. Agrees with the paired Monte Carlo permutation test on all nine screen arms.
# F remains the fallback when the identical-config reference pair is unavailable.
REF_A, REF_B, N_BOOT, N_PERM = "baseline", "baseline__rerun", 20000, 20000
try:
    import numpy as np
    if {RUN_A, RUN_B} == {REF_A, REF_B}:
        raise ValueError("this contrast IS the reference pair; it defines the floor")
    Ra, _ = load(REF_A); Rb, _ = load(REF_B)
    ref = {Ra[i]["uid"]: ov(Ra[i]) - ov(Rb[i]) for i in sorted(set(Ra) & set(Rb))}
    cur = {A[i]["uid"]: ov(A[i]) - ov(B[i]) for i in I}
    u = sorted(set(ref) & set(cur))
    if len(u) < 30: raise ValueError(f"only {len(u)} rows shared with the reference pair")
    X = np.array([cur[k] for k in u]); Y = np.array([ref[k] for k in u]); m = len(u)
    if X.var(ddof=1) == 0 or Y.var(ddof=1) == 0: raise ValueError("degenerate: zero variance")
    ratio = X.var(ddof=1) / Y.var(ddof=1)
    idx = np.random.default_rng(0).integers(0, m, size=(N_BOOT, m))
    rb  = X[idx].var(axis=1, ddof=1) / Y[idx].var(axis=1, ddof=1)
    p_b = min(2*min((rb <= 1).mean(), (rb >= 1).mean()), 1.0)
    lo, hi = np.percentile(rb, [2.5, 97.5])
    print(f"scatter vs floor       : var ratio {ratio:.2f} 95%CI [{lo:.2f}, {hi:.2f}] "
          f"paired-bootstrap p={p_b:.4f} on {m} rows -> "
          f"{'re-rolls rows beyond generation noise' if p_b < 0.05 else 'no excess scatter'}")
    # Monte Carlo test of the same hypothesis: this arm and the rerun are exchangeable given the
    # shared baseline. Both delta series are measured against the same run, so the swap is
    # per row (d_arm_i <-> d_ref_i), which preserves that pairing; a pooled shuffle does not
    # and returns materially different p-values. Statistic is |log| of the variance ratio.
    _sw = np.random.default_rng(0).random((N_PERM, m)) < 0.5
    _Xp, _Yp = np.where(_sw, Y, X), np.where(_sw, X, Y)
    _rp = _Xp.var(axis=1, ddof=1) / _Yp.var(axis=1, ddof=1)
    p_perm = (1 + int((np.abs(np.log(_rp)) >= abs(math.log(ratio))).sum())) / (N_PERM + 1)
    print(f"{'':23s}  Monte Carlo permutation p={p_perm:.4f} ({N_PERM} random paired swaps) -> "
          f"{'re-rolls rows beyond generation noise' if p_perm < 0.05 else 'no excess scatter'}")
except (FileNotFoundError, ValueError) as e:
    F   = (sd*sd) / (SIGMA_D_REF*SIGMA_D_REF)
    p_F = 2*min(fdist.cdf(F, df, df), 1 - fdist.cdf(F, df, df))
    print(f"scatter vs floor       : F={F:.2f} p={p_F:.2e} [independent F, normality assumed: {e}] -> "
          f"{'re-rolls rows' if p_F < 0.05 else 'no excess scatter'}")
print(f"\nDECISION: p in the table above. p<0.05 = the runs differ; read the sign of mean for direction.")
print(f"MDE (planning quantity, NOT a pass/fail rule) for THIS contrast at n={n}: {thr:.3f}")
print(f"  = {Z:.2f} x {sd:.3f} / sqrt({n})")
print(f"  (using the floor {SIGMA_D_REF} instead would give {Z*SIGMA_D_REF/math.sqrt(n):.3f} — an understatement)")
print(f"  observed |mean Overall| = {abs(st.mean(d)):.3f} -> a true effect below ~{thr:.3f} is tested with under 80% power at this n")

metric       mean      SD      SE             95% CI       t       p
PF         +0.086   1.066   0.061   [-0.034, +0.206]   +1.40   0.161
SC         -0.003   1.543   0.089   [-0.177, +0.171]   -0.04   0.970
Overall    +0.037   1.093   0.063   [-0.087, +0.160]   +0.58   0.560

sigma_d (Overall)      = 1.093   95% CI [1.012, 1.187]
identical-config floor = 0.966   (re-running the SAME config)
judge-only floor       = 0.512   (sqrt(2) x 0.362)
scatter vs floor       : var ratio 1.28 95%CI [0.77, 2.12] paired-bootstrap p=0.3305 on 302 rows -> no excess scatter
                         Monte Carlo permutation p=0.3400 (20000 random paired swaps) -> no excess scatter

DECISION: p in the table above. p<0.05 = the runs differ; read the sign of mean for direction.
MDE (planning quantity, NOT a pass/fail rule) for THIS contrast at n=302: 0.176
  = 2.80 x 1.093 / sqrt(302)
  (using the floor 0.966 instead would give 0.156 — an understatement)
  observed |mean Overall| = 0.037 -> a true effect bel

## 4. Equivalence (TOST) against the pre-registered bound

In [ ]:
from scipy.stats import t as tdist

mean, se = st.mean(d), sd/math.sqrt(n)
t1 = (mean - (-EQUIV_BOUND)) / se          # H0: delta <= -bound
t2 = (mean -  EQUIV_BOUND)  / se           # H0: delta >= +bound
p_tost = max(1 - tdist.cdf(t1, df), tdist.cdf(t2, df))
c_lo, c_hi = mean - tdist.ppf(0.95, df)*se, mean + tdist.ppf(0.95, df)*se   # 90% CI = TOST at a=.05

print(f"pre-registered bound   : +/-{EQUIV_BOUND}")
print(f"mean Overall delta     : {mean:+.3f}")
print(f"90% CI                 : [{c_lo:+.3f}, {c_hi:+.3f}]")
print(f"TOST p                 : {p_tost:.3f}")
# TOST rejects only if BOTH one-sided tests reject: power at true delta 0 is 2*Phi(.)-1, not Phi(.)
print(f"TOST power at n={n}, sigma_d={sd:.3f}: {max(2*norm.cdf(EQUIV_BOUND/se - 1.645) - 1, 0.0):.0%}")
print(f"smallest bound provable here at 80% TOST power: {(1.645 + norm.ppf(0.90))*se:.3f}")

inside  = c_lo > -EQUIV_BOUND and c_hi < EQUIV_BOUND
outside = c_lo >  EQUIV_BOUND or  c_hi < -EQUIV_BOUND
print("\nVERDICT: ", end="")
if outside:
    print(f"DIFFER — 90% CI lies entirely outside +/-{EQUIV_BOUND}.")
elif inside and DATASET != "test":
    print(f"NULL (screen). TOST passes (p={p_tost:.3f}) but equivalence is claimed only on the "
          f"test split; at n={n} the power of this contrast is "
          f"{max(2*norm.cdf(EQUIV_BOUND/se - 1.645) - 1, 0.0):.0%}.")
elif inside:
    print(f"EQUIVALENT — 90% CI lies entirely within +/-{EQUIV_BOUND} (TOST p={p_tost:.3f}).")
else:
    print(f"INCONCLUSIVE — 90% CI straddles +/-{EQUIV_BOUND}. Not evidence of equivalence.")

pre-registered bound   : +/-0.15
mean Overall delta     : +0.037
90% CI                 : [-0.067, +0.140]
TOST p                 : 0.036
TOST power at n=302, sigma_d=1.093: 54%
smallest bound provable here at 80% TOST power: 0.184

VERDICT: NULL (screen). TOST passes (p=0.036) but equivalence is claimed only on the test split; at n=302 the power of this contrast is 54%.


## 5. Strata and the geometric-mean floor

In [ ]:
# only pre-generation variables. Stratifying a delta by baseline score returns a gain from
# regression to the mean alone: on an identical config, rows with Overall<4 rise +1.44.
for key in ["n_refs", "ref_kind"]:
    g = defaultdict(list)
    for i in I: g[A[i][key]].append(ov(A[i]) - ov(B[i]))
    print(f"by {key}:")
    for k, v in sorted(g.items(), key=str):
        print(f"   {str(k):8s} n={len(v):3d}  mean {st.mean(v):+.3f}  SD {st.stdev(v) if len(v)>1 else float('nan'):.3f}")

z = [i for i in I if A[i]["SC"] == 0 or B[i]["SC"] == 0]
print(f"\nrows where either run scored SC=0: {len(z)}/{n} ({len(z)/n:.1%})")
if z:
    keep = [i for i in I if i not in z]
    dk = [ov(A[i]) - ov(B[i]) for i in keep]
    _, pk = ttest_rel([ov(A[i]) for i in keep], [ov(B[i]) for i in keep])
    print(f"  SC=0 forces Overall to exactly 0 regardless of PF (geometric mean), so those rows")
    print(f"  carry outsized leverage on any mean they fall in.")
    print(f"  excluding them: n={len(keep)} mean {st.mean(dk):+.3f} SD {st.stdev(dk):.3f} p {pk:.3f}")

by n_refs:
   1        n=152  mean -0.006  SD 1.224
   2        n=120  mean +0.031  SD 1.008
   3        n= 30  mean +0.277  SD 0.598
by ref_kind:
   all_up   n=119  mean -0.098  SD 1.556
   mixed    n= 73  mean +0.188  SD 0.713
   no_up    n=110  mean +0.082  SD 0.551

rows where either run scored SC=0: 4/302 (1.3%)
  SC=0 forces Overall to exactly 0 regardless of PF (geometric mean), so those rows
  carry outsized leverage on any mean they fall in.
  excluding them: n=298 mean +0.040 SD 1.047 p 0.513
